In [3]:
# Lire les données brutes depuis la couche Bronze

bronze_path = "abfss://healthcare@adlgenstorage.dfs.core.windows.net/Bronze/healthcare_dataset_raw_imperfect.csv"

df_bronze = (
    spark.read
    .option("header", "true")
    .option("inferSchema", "true")
    .csv(bronze_path)
)

print("Nombre de lignes Bronze :", df_bronze.count())

StatementMeta(sparkhealcare, 2, 4, Finished, Available, Finished, False)

Nombre de lignes Bronze : 56150


In [4]:
# Supprimer les lignes complètement identiques

df_silver = df_bronze.dropDuplicates()

print("Lignes Bronze :", df_bronze.count())
print("Lignes après suppression des doublons :", df_silver.count())

StatementMeta(sparkhealcare, 2, 5, Finished, Available, Finished, False)

Lignes Bronze : 56150
Lignes après suppression des doublons : 55173


In [5]:
# Compter les valeurs NULL restantes dans chaque colonne après suppression des doublons

from pyspark.sql.functions import col, sum

df_silver.select([
    sum(col(c).isNull().cast("int")).alias(c)
    for c in df_silver.columns
]).show()

StatementMeta(sparkhealcare, 2, 6, Finished, Available, Finished, False)

+----+---+------+----------+-----------------+-----------------+------+--------+------------------+--------------+-----------+--------------+--------------+----------+------------+
|Name|Age|Gender|Blood Type|Medical Condition|Date of Admission|Doctor|Hospital|Insurance Provider|Billing Amount|Room Number|Admission Type|Discharge Date|Medication|Test Results|
+----+---+------+----------+-----------------+-----------------+------+--------+------------------+--------------+-----------+--------------+--------------+----------+------------+
|   0|  0|     0|       397|                0|                0|   500|       0|               840|           297|          0|             0|           179|       686|         586|
+----+---+------+----------+-----------------+-----------------+------+--------+------------------+--------------+-----------+--------------+--------------+----------+------------+



In [6]:
# Nettoyer les espaces et standardiser la casse des colonnes texte importantes

from pyspark.sql.functions import trim, initcap, upper

df_silver = (
    df_silver
    .withColumn("Name", initcap(trim(col("Name"))))
    .withColumn("Gender", initcap(trim(col("Gender"))))
    .withColumn("Blood Type", upper(trim(col("Blood Type"))))
    .withColumn("Medical Condition", initcap(trim(col("Medical Condition"))))
    .withColumn("Doctor", initcap(trim(col("Doctor"))))
    .withColumn("Hospital", initcap(trim(col("Hospital"))))
    .withColumn("Insurance Provider", initcap(trim(col("Insurance Provider"))))
    .withColumn("Admission Type", initcap(trim(col("Admission Type"))))
    .withColumn("Medication", initcap(trim(col("Medication"))))
    .withColumn("Test Results", initcap(trim(col("Test Results"))))
)

StatementMeta(sparkhealcare, 2, 7, Finished, Available, Finished, False)

In [7]:
# Vérifier quelques colonnes après standardisation

df_silver.select(
    "Name",
    "Gender",
    "Blood Type",
    "Medical Condition",
    "Admission Type"
).show(20, truncate=False)

StatementMeta(sparkhealcare, 2, 8, Finished, Available, Finished, False)

+---------------------+------+----------+-----------------+--------------+
|Name                 |Gender|Blood Type|Medical Condition|Admission Type|
+---------------------+------+----------+-----------------+--------------+
|Dawn Black           |Female|B+        |Diabetes         |Emergency     |
|Bobby Davis          |Male  |A-        |Hypertension     |Urgent        |
|Mrs. Tiffany Dennis  |Male  |O-        |Diabetes         |Urgent        |
|Dana Stone           |Male  |A-        |Obesity          |Elective      |
|Thomas Conner        |Male  |A-        |Diabetes         |Elective      |
|Wayne Diaz Md        |Male  |B+        |Arthritis        |Urgent        |
|Stacy Baxter         |Female|O-        |Cancer           |Emergency     |
|Heather Morgan       |Male  |B+        |Asthma           |Emergency     |
|Joshua Galloway      |Male  |O+        |Asthma           |Elective      |
|Megan Phillips       |Female|O-        |Cancer           |Emergency     |
|Kimberly Lee         |Fe

In [8]:
# Vérifier que les catégories Gender sont maintenant uniformisées

df_silver.groupBy("Gender").count().show()

StatementMeta(sparkhealcare, 2, 9, Finished, Available, Finished, False)

+-------+-----+
| Gender|count|
+-------+-----+
| Female|27511|
|Unknown|   45|
|      X|   43|
|   Male|27542|
|      ?|   32|
+-------+-----+



In [9]:
# Standardiser les valeurs invalides de Gender
# Seuls Male et Female sont conservés.
# Les autres valeurs deviennent Unknown.

from pyspark.sql.functions import when, col

df_silver = df_silver.withColumn(
    "Gender",
    when(
        col("Gender").isin("Male", "Female"),
        col("Gender")
    ).otherwise("Unknown")
)

StatementMeta(sparkhealcare, 2, 10, Finished, Available, Finished, False)

In [10]:
# Vérifier le résultat après nettoyage de Gender

df_silver.groupBy("Gender").count().show()

StatementMeta(sparkhealcare, 2, 11, Finished, Available, Finished, False)

+-------+-----+
| Gender|count|
+-------+-----+
| Female|27511|
|Unknown|  120|
|   Male|27542|
+-------+-----+



In [11]:
# Vérifier les valeurs des principales colonnes catégorielles
# avant d'appliquer les règles de nettoyage.

colonnes_a_verifier = [
    "Blood Type",
    "Medical Condition",
    "Admission Type",
    "Test Results"
]

for c in colonnes_a_verifier:
    print(f"\n===== {c} =====")
    df_silver.groupBy(c) \
        .count() \
        .orderBy("count", ascending=False) \
        .show(truncate=False)

StatementMeta(sparkhealcare, 2, 12, Finished, Available, Finished, False)


===== Blood Type =====
+----------+-----+
|Blood Type|count|
+----------+-----+
|A-        |6872 |
|A+        |6871 |
|AB-       |6863 |
|B+        |6863 |
|AB+       |6856 |
|B-        |6854 |
|O+        |6824 |
|O-        |6770 |
|NULL      |397  |
|NAN       |3    |
+----------+-----+


===== Medical Condition =====
+-----------------+-----+
|Medical Condition|count|
+-----------------+-----+
|Arthritis        |9250 |
|Diabetes         |9248 |
|Hypertension     |9189 |
|Obesity          |9184 |
|Cancer           |9170 |
|Asthma           |9132 |
+-----------------+-----+


===== Admission Type =====
+--------------+-----+
|Admission Type|count|
+--------------+-----+
|Elective      |18501|
|Urgent        |18417|
|Emergency     |18115|
|?             |53   |
|Unknown       |48   |
|Walk-in       |39   |
+--------------+-----+


===== Test Results =====
+------------+-----+
|Test Results|count|
+------------+-----+
|Abnormal    |18196|
|Normal      |18059|
|Inconclusive|17955|
|NULL 

In [12]:
# Remplacer les NULL de certaines colonnes texte par "Unknown"
# afin de conserver les autres informations utiles de la ligne.

colonnes_unknown = [
    "Gender",
    "Blood Type",
    "Medical Condition",
    "Doctor",
    "Hospital",
    "Insurance Provider",
    "Admission Type",
    "Medication",
    "Test Results"
]

df_silver = df_silver.fillna(
    "Unknown",
    subset=colonnes_unknown
)

StatementMeta(sparkhealcare, 2, 13, Finished, Available, Finished, False)

In [13]:
# Recompter les NULL après le premier traitement
# pour identifier les colonnes qui nécessitent une autre stratégie.

from pyspark.sql.functions import col, sum

df_silver.select([
    sum(col(c).isNull().cast("int")).alias(c)
    for c in df_silver.columns
]).show()

StatementMeta(sparkhealcare, 2, 14, Finished, Available, Finished, False)

+----+---+------+----------+-----------------+-----------------+------+--------+------------------+--------------+-----------+--------------+--------------+----------+------------+
|Name|Age|Gender|Blood Type|Medical Condition|Date of Admission|Doctor|Hospital|Insurance Provider|Billing Amount|Room Number|Admission Type|Discharge Date|Medication|Test Results|
+----+---+------+----------+-----------------+-----------------+------+--------+------------------+--------------+-----------+--------------+--------------+----------+------------+
|   0|  0|     0|         0|                0|                0|     0|       0|                 0|           297|          0|             0|           179|         0|           0|
+----+---+------+----------+-----------------+-----------------+------+--------+------------------+--------------+-----------+--------------+--------------+----------+------------+



In [14]:
# Remplacer les Room Number NULL par -1
# -1 signifie : numéro de chambre inconnu ou non renseigné

df_silver = df_silver.fillna(
    {"Room Number": -1}
)

StatementMeta(sparkhealcare, 2, 15, Finished, Available, Finished, False)

In [15]:
# Vérifier le résultat après traitement

df_silver.groupBy("Room Number") \
    .count() \
    .orderBy("Room Number") \
    .show(20)

StatementMeta(sparkhealcare, 2, 16, Finished, Available, Finished, False)

+-----------+-----+
|Room Number|count|
+-----------+-----+
|        -10|   35|
|          0|   43|
|        101|  120|
|        102|  158|
|        103|  136|
|        104|  172|
|        105|  118|
|        106|  134|
|        107|  137|
|        108|  137|
|        109|  137|
|        110|  132|
|        111|  136|
|        112|  157|
|        113|  132|
|        114|  133|
|        115|  128|
|        116|  135|
|        117|  124|
|        118|  127|
+-----------+-----+
only showing top 20 rows



In [16]:
# Examiner quelques lignes dont la date d'admission est manquante
# pour décider comment les traiter correctement

df_silver.filter(
    col("Date of Admission").isNull()
).select(
    "Name",
    "Date of Admission",
    "Discharge Date",
    "Admission Type",
    "Hospital",
    "Room Number"
).show(20, truncate=False)

StatementMeta(sparkhealcare, 2, 17, Finished, Available, Finished, False)

+----+-----------------+--------------+--------------+--------+-----------+
|Name|Date of Admission|Discharge Date|Admission Type|Hospital|Room Number|
+----+-----------------+--------------+--------------+--------+-----------+
+----+-----------------+--------------+--------------+--------+-----------+



In [17]:
df_silver.filter(
    col("Date of Admission").isNull()
)

StatementMeta(sparkhealcare, 2, 18, Finished, Available, Finished, False)

DataFrame[Name: string, Age: int, Gender: string, Blood Type: string, Medical Condition: string, Date of Admission: string, Doctor: string, Hospital: string, Insurance Provider: string, Billing Amount: double, Room Number: int, Admission Type: string, Discharge Date: string, Medication: string, Test Results: string]

In [18]:
# Vérifier combien de vrais NULL restent dans Date of Admission

df_silver.filter(
    col("Date of Admission").isNull()
).count()

StatementMeta(sparkhealcare, 2, 19, Finished, Available, Finished, False)

0

In [19]:
# Vérifier combien de vrais NULL restent dans Date of Admission

df_silver.filter(
    col("Date of Admission").isNull()
).count()

StatementMeta(sparkhealcare, 2, 20, Finished, Available, Finished, False)

0

In [20]:
# Vérifier les valeurs NULL restantes dans toutes les colonnes
# Cela permet de contrôler la qualité des données après le nettoyage Silver.

from pyspark.sql.functions import col, sum

df_silver.select([
    sum(col(c).isNull().cast("int")).alias(c)
    for c in df_silver.columns
]).show()

StatementMeta(sparkhealcare, 2, 21, Finished, Available, Finished, False)

+----+---+------+----------+-----------------+-----------------+------+--------+------------------+--------------+-----------+--------------+--------------+----------+------------+
|Name|Age|Gender|Blood Type|Medical Condition|Date of Admission|Doctor|Hospital|Insurance Provider|Billing Amount|Room Number|Admission Type|Discharge Date|Medication|Test Results|
+----+---+------+----------+-----------------+-----------------+------+--------+------------------+--------------+-----------+--------------+--------------+----------+------------+
|   0|  0|     0|         0|                0|                0|     0|       0|                 0|           297|          0|             0|           179|         0|           0|
+----+---+------+----------+-----------------+-----------------+------+--------+------------------+--------------+-----------+--------------+--------------+----------+------------+



In [21]:
# Examiner les lignes où Date of Admission est NULL
# pour comprendre les données avant de décider du traitement.

df_silver.filter(
    col("Date of Admission").isNull()
).select(
    "Name",
    "Date of Admission",
    "Discharge Date",
    "Admission Type",
    "Hospital"
).show(20, truncate=False)

StatementMeta(sparkhealcare, 2, 22, Finished, Available, Finished, False)

+----+-----------------+--------------+--------------+--------+
|Name|Date of Admission|Discharge Date|Admission Type|Hospital|
+----+-----------------+--------------+--------------+--------+
+----+-----------------+--------------+--------------+--------+



In [22]:
# Compter exactement les lignes concernées

date_admission_null = df_silver.filter(
    col("Date of Admission").isNull()
).count()

print("Nombre de Date of Admission NULL :", date_admission_null)

StatementMeta(sparkhealcare, 2, 23, Finished, Available, Finished, False)

Nombre de Date of Admission NULL : 0


In [23]:
# Compter les NULL actuellement présents dans Discharge Date
# Cela permet de savoir si cette colonne nécessite encore un nettoyage.

discharge_null = df_silver.filter(
    col("Discharge Date").isNull()
).count()

print("Nombre de Discharge Date NULL :", discharge_null)

StatementMeta(sparkhealcare, 2, 24, Finished, Available, Finished, False)

Nombre de Discharge Date NULL : 179


In [24]:
# Examiner les lignes où Discharge Date est NULL
# avant de décider si elles doivent être conservées, corrigées ou supprimées.

df_silver.filter(
    col("Discharge Date").isNull()
).select(
    "Name",
    "Date of Admission",
    "Discharge Date",
    "Admission Type",
    "Hospital",
    "Medical Condition"
).show(20, truncate=False)

StatementMeta(sparkhealcare, 2, 25, Finished, Available, Finished, False)

+-------------------+-----------------+--------------+--------------+------------------------------+-----------------+
|Name               |Date of Admission|Discharge Date|Admission Type|Hospital                      |Medical Condition|
+-------------------+-----------------+--------------+--------------+------------------------------+-----------------+
|Tara Chavez        |2022-01-11       |NULL          |Emergency     |Martin Phillips Glover, And   |Hypertension     |
|Barbara Miller     |2023-02-21       |NULL          |Urgent        |Combs Sons And                |Asthma           |
|Nicole Lopez       |2021-07-06       |NULL          |Emergency     |Berg-avery                    |Hypertension     |
|Lucas Schroeder    |2020-09-18       |NULL          |Elective      |James Inc                     |Obesity          |
|Robin Patterson    |2023-02-09       |NULL          |Urgent        |Chen-castillo                 |Hypertension     |
|James Chandler     |2020-01-15       |NULL     

In [25]:
# Vérifier combien de valeurs NULL restent dans Discharge Date
# Une date de sortie NULL peut signifier qu'un patient n'est pas encore sorti.

discharge_null = df_silver.filter(
    col("Discharge Date").isNull()
).count()

print("Nombre de Discharge Date NULL :", discharge_null)

StatementMeta(sparkhealcare, 2, 26, Finished, Available, Finished, False)

Nombre de Discharge Date NULL : 179


In [26]:
# Afficher quelques lignes ayant une date de sortie manquante
# pour comprendre si ces NULL sont cohérents avec les données.

df_silver.filter(
    col("Discharge Date").isNull()
).select(
    "Name",
    "Date of Admission",
    "Discharge Date",
    "Admission Type",
    "Hospital",
    "Medical Condition"
).show(20, truncate=False)

StatementMeta(sparkhealcare, 2, 27, Finished, Available, Finished, False)

+-------------------+-----------------+--------------+--------------+------------------------------+-----------------+
|Name               |Date of Admission|Discharge Date|Admission Type|Hospital                      |Medical Condition|
+-------------------+-----------------+--------------+--------------+------------------------------+-----------------+
|Tara Chavez        |2022-01-11       |NULL          |Emergency     |Martin Phillips Glover, And   |Hypertension     |
|Barbara Miller     |2023-02-21       |NULL          |Urgent        |Combs Sons And                |Asthma           |
|Nicole Lopez       |2021-07-06       |NULL          |Emergency     |Berg-avery                    |Hypertension     |
|Lucas Schroeder    |2020-09-18       |NULL          |Elective      |James Inc                     |Obesity          |
|Robin Patterson    |2023-02-09       |NULL          |Urgent        |Chen-castillo                 |Hypertension     |
|James Chandler     |2020-01-15       |NULL     

In [27]:
# Contrôle métier :
# vérifier qu'aucune date de sortie n'est antérieure à la date d'admission.

from pyspark.sql.functions import col

invalid_dates = df_silver.filter(
    col("Discharge Date") < col("Date of Admission")
).count()

print("Nombre de dates incohérentes :", invalid_dates)

StatementMeta(sparkhealcare, 2, 28, Finished, Available, Finished, False)

Nombre de dates incohérentes : 1017


In [28]:
# Afficher quelques lignes avec des dates incohérentes
# pour analyser le problème avant d'appliquer une correction.

df_silver.filter(
    col("Discharge Date") < col("Date of Admission")
).select(
    "Name",
    "Date of Admission",
    "Discharge Date",
    "Admission Type",
    "Hospital",
    "Medical Condition"
).show(20, truncate=False)

StatementMeta(sparkhealcare, 2, 29, Finished, Available, Finished, False)

+--------------------+-----------------+--------------+--------------+-----------------------------+-----------------+
|Name                |Date of Admission|Discharge Date|Admission Type|Hospital                     |Medical Condition|
+--------------------+-----------------+--------------+--------------+-----------------------------+-----------------+
|Thomas Dennis       |Jan 28 2024      |2024-02-21    |Elective      |Mckinney-edwards             |Asthma           |
|Amy Tapia           |2020-08-27       |2020-08-23    |Urgent        |Brown Ltd                    |Obesity          |
|Andrew Perez        |2023-06-21       |2023-06-16    |Elective      |Simmons Thompson, And Smith  |Hypertension     |
|Holly Blackburn     |2023-05-24       |06/05/2023    |Urgent        |Boyd-knight                  |Arthritis        |
|Debbie Miller       |Feb 05 2023      |2023-02-14    |Urgent        |Oneal-morris                 |Arthritis        |
|Bonnie Simmons      |2019-07-04       |07/16/20

In [29]:
# DATA QUALITY :
# Afficher quelques lignes où la date de sortie
# est antérieure à la date d'admission.

df_silver.filter(
    col("Discharge Date") < col("Date of Admission")
).select(
    "Name",
    "Date of Admission",
    "Discharge Date",
    "Admission Type",
    "Hospital"
).show(20, truncate=False)

StatementMeta(sparkhealcare, 2, 30, Finished, Available, Finished, False)

+--------------------+-----------------+--------------+--------------+-----------------------------+
|Name                |Date of Admission|Discharge Date|Admission Type|Hospital                     |
+--------------------+-----------------+--------------+--------------+-----------------------------+
|Thomas Dennis       |Jan 28 2024      |2024-02-21    |Elective      |Mckinney-edwards             |
|Amy Tapia           |2020-08-27       |2020-08-23    |Urgent        |Brown Ltd                    |
|Andrew Perez        |2023-06-21       |2023-06-16    |Elective      |Simmons Thompson, And Smith  |
|Holly Blackburn     |2023-05-24       |06/05/2023    |Urgent        |Boyd-knight                  |
|Debbie Miller       |Feb 05 2023      |2023-02-14    |Urgent        |Oneal-morris                 |
|Bonnie Simmons      |2019-07-04       |07/16/2019    |Emergency     |Smith, Shaw Stevens And      |
|Dana Miller         |2021-05-27       |18-06-2021    |Emergency     |Alvarez-valentine    

In [30]:
# DATA CLEANING :
# Si la date de sortie est antérieure à la date d'admission,
# on remplace uniquement la date de sortie par NULL.
# On conserve toutes les autres informations de la ligne.

from pyspark.sql.functions import col, when

df_silver = df_silver.withColumn(
    "Discharge Date",
    when(
        col("Discharge Date") < col("Date of Admission"),
        None
    ).otherwise(col("Discharge Date"))
)

StatementMeta(sparkhealcare, 2, 31, Finished, Available, Finished, False)

In [31]:
# DATA QUALITY :
# Vérifier qu'il ne reste plus de date de sortie
# antérieure à la date d'admission.

invalid_dates_after = df_silver.filter(
    col("Discharge Date") < col("Date of Admission")
).count()

print("Dates incohérentes après nettoyage :", invalid_dates_after)

StatementMeta(sparkhealcare, 2, 32, Finished, Available, Finished, False)

Dates incohérentes après nettoyage : 0


In [32]:
# DATA QUALITY FINAL
# Vérifier l'état du dataset après toutes les transformations Silver.

from pyspark.sql.functions import col, sum

print("===== CONTROLE FINAL SILVER =====")

# 1. Nombre total de lignes
print("Nombre total de lignes :", df_silver.count())

# 2. Nombre de doublons
duplicates = df_silver.count() - df_silver.dropDuplicates().count()
print("Nombre de doublons :", duplicates)

# 3. Valeurs NULL par colonne
print("\nValeurs NULL par colonne :")

df_silver.select([
    sum(col(c).isNull().cast("int")).alias(c)
    for c in df_silver.columns
]).show(truncate=False)

# 4. Vérification des dates incohérentes
invalid_dates = df_silver.filter(
    col("Discharge Date") < col("Date of Admission")
).count()

print("Dates incohérentes :", invalid_dates)

StatementMeta(sparkhealcare, 2, 33, Finished, Available, Finished, False)

===== CONTROLE FINAL SILVER =====
Nombre total de lignes : 55173
Nombre de doublons : 67

Valeurs NULL par colonne :
+----+---+------+----------+-----------------+-----------------+------+--------+------------------+--------------+-----------+--------------+--------------+----------+------------+
|Name|Age|Gender|Blood Type|Medical Condition|Date of Admission|Doctor|Hospital|Insurance Provider|Billing Amount|Room Number|Admission Type|Discharge Date|Medication|Test Results|
+----+---+------+----------+-----------------+-----------------+------+--------+------------------+--------------+-----------+--------------+--------------+----------+------------+
|0   |0  |0     |0         |0                |0                |0     |0       |0                 |297           |0          |0             |1196          |0         |0           |
+----+---+------+----------+-----------------+-----------------+------+--------+------------------+--------------+-----------+--------------+--------------+---

In [33]:
# Supprimer les lignes complètement dupliquées
# On conserve une seule occurrence de chaque ligne identique.

df_silver = df_silver.dropDuplicates()

print("Nombre de lignes après suppression des doublons :", df_silver.count())

StatementMeta(sparkhealcare, 2, 34, Finished, Available, Finished, False)

Nombre de lignes après suppression des doublons : 55106


In [34]:
# Remplacer les assurances manquantes par "Unknown"
# Cela permet de conserver les patients même si leur assurance n'est pas renseignée.

df_silver = df_silver.fillna({
    "Insurance Provider": "Unknown"
})

# Vérifier le résultat
df_silver.groupBy("Insurance Provider").count().orderBy("count").show(
    truncate=False
)

StatementMeta(sparkhealcare, 2, 35, Finished, Available, Finished, False)

+------------------+-----+
|Insurance Provider|count|
+------------------+-----+
|Nan               |4    |
|?                 |98   |
|N/a               |128  |
|Unknown           |964  |
|Aetna             |10613|
|Blue Cross        |10720|
|Unitedhealthcare  |10797|
|Medicare          |10833|
|Cigna             |10949|
+------------------+-----+



In [35]:
# Vérifier les NULL après le traitement de Insurance Provider

from pyspark.sql.functions import col, sum

df_silver.select([
    sum(col(c).isNull().cast("int")).alias(c)
    for c in df_silver.columns
]).show(truncate=False)

StatementMeta(sparkhealcare, 2, 36, Finished, Available, Finished, False)

+----+---+------+----------+-----------------+-----------------+------+--------+------------------+--------------+-----------+--------------+--------------+----------+------------+
|Name|Age|Gender|Blood Type|Medical Condition|Date of Admission|Doctor|Hospital|Insurance Provider|Billing Amount|Room Number|Admission Type|Discharge Date|Medication|Test Results|
+----+---+------+----------+-----------------+-----------------+------+--------+------------------+--------------+-----------+--------------+--------------+----------+------------+
|0   |0  |0     |0         |0                |0                |0     |0       |0                 |297           |0          |0             |1195          |0         |0           |
+----+---+------+----------+-----------------+-----------------+------+--------+------------------+--------------+-----------+--------------+--------------+----------+------------+



In [36]:
# Afficher quelques lignes où Insurance Provider est NULL
# afin de comprendre le problème avant de le corriger.

df_silver.filter(
    col("Insurance Provider").isNull()
).select(
    "Name",
    "Insurance Provider",
    "Medical Condition",
    "Hospital"
).show(20, truncate=False)

StatementMeta(sparkhealcare, 2, 37, Finished, Available, Finished, False)

+----+------------------+-----------------+--------+
|Name|Insurance Provider|Medical Condition|Hospital|
+----+------------------+-----------------+--------+
+----+------------------+-----------------+--------+



In [37]:
df_silver = df_silver.fillna(
    {"Insurance Provider": "Unknown"}
)

StatementMeta(sparkhealcare, 2, 38, Finished, Available, Finished, False)

In [38]:
# Remplacer les valeurs NULL de Insurance Provider par "Unknown"
# On conserve ainsi les lignes sans inventer le nom d'une assurance.

from pyspark.sql.functions import col, when

df_silver = df_silver.withColumn(
    "Insurance Provider",
    when(
        col("Insurance Provider").isNull(),
        "Unknown"
    ).otherwise(col("Insurance Provider"))
)

StatementMeta(sparkhealcare, 2, 39, Finished, Available, Finished, False)

In [39]:
# Vérifier qu'il ne reste plus de NULL dans Insurance Provider

insurance_null = df_silver.filter(
    col("Insurance Provider").isNull()
).count()

print("Insurance Provider NULL :", insurance_null)

StatementMeta(sparkhealcare, 2, 40, Finished, Available, Finished, False)

Insurance Provider NULL : 0


In [42]:
# Standardiser les noms des colonnes pour la couche Silver
# Exemple : "Blood Type" devient "blood_type"

for old_name in df_silver.columns:
    new_name = (
        old_name.strip()
        .lower()
        .replace(" ", "_")
    )

    df_silver = df_silver.withColumnRenamed(old_name, new_name)

# Vérifier les nouveaux noms
print(df_silver.columns)

StatementMeta(sparkhealcare, 2, 43, Finished, Available, Finished, False)

['name', 'age', 'gender', 'blood_type', 'medical_condition', 'date_of_admission', 'doctor', 'hospital', 'insurance_provider', 'billing_amount', 'room_number', 'admission_type', 'discharge_date', 'medication', 'test_results']


In [43]:
silver_path = "abfss://healthcare@adlgenstorage.dfs.core.windows.net/Silver/healthcare_clean/"

StatementMeta(sparkhealcare, 2, 44, Finished, Available, Finished, False)

In [44]:
df_silver.write \
    .format("delta") \
    .mode("overwrite") \
    .save(silver_path)

StatementMeta(sparkhealcare, 2, 45, Finished, Available, Finished, False)